# Imports

In [ ]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder) without this

In [ ]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *
import numpy as np
from Queries import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

# Ensure uncached tables

In [ ]:
spark.catalog.uncacheTable("default.air_quality")
spark.catalog.uncacheTable("default.taxi_trips")
spark.catalog.uncacheTable("default.taxi_zone_lookup")
spark.catalog.uncacheTable("default.weather")

# Ensure no auto broadcast

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# reset the broadcast threshold to default value
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)  # 10 MB

# Regular queries
Make sure to manually check no broadcast occurs in execution plan

### Query 2.1

In [13]:
result = spark.sql(query_2_1(broadcast=False))
result.show()
result.explain(mode="formatted")

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|            Kips Bay|    1|    32984|
|                SoHo|    1|    20197|
|Upper East Side N...|   12|        1|
|Breezy Point/Fort...|    1|        6|
|    Bensonhurst West|    1|      153|
|       East New York|    1|      938|
|Flushing Meadows-...|    1|      510|
|        Battery Park|    1|      875|
|        Borough Park|    1|      237|
| Grymes Hill/Clifton|    1|        2|
|          Kensington|    1|      127|
|         Hunts Point|    1|      105|
|       Melrose South|    1|      289|
|       Prospect Park|    1|       53|
|          Pelham Bay|    1|       47|
| UN/Turtle Bay South|    1|    33595|
|Washington Height...|    1|      495|
|       Willets Point|    1|       14|
|Downtown Brooklyn...|    1|     1393|
|    Inwood Hill Park|    1|       22|
+--------------------+-----+---------+
only showing top 20 rows
== Physical Plan ==
AdaptiveSparkPlan (

### Query 2.2

In [14]:
result = spark.sql(query_2_2(broadcast=False))
result.show()
result.explain(mode="formatted")

+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 424773| 3.470743355446981|
|  zero_or_null|2539795|3.6824345463125323|
+--------------+-------+------------------+

== Physical Plan ==
AdaptiveSparkPlan (13)
+- HashAggregate (12)
   +- Exchange (11)
      +- HashAggregate (10)
         +- Project (9)
            +- SortMergeJoin LeftOuter (8)
               :- Sort (3)
               :  +- Exchange (2)
               :     +- Scan parquet spark_catalog.default.taxi_trips (1)
               +- Sort (7)
                  +- Exchange (6)
                     +- Filter (5)
                        +- Scan parquet spark_catalog.default.weather (4)


(1) Scan parquet spark_catalog.default.taxi_trips
Output [2]: [pu_datetime#6416, trip_distance#6421]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/mac/Documents/dev/ID2221/dic/spark_project/spark-warehouse/taxi_trips]
ReadSchema: stru

### Query 2.3

In [15]:
res = spark.sql(query_2_3(broadcast=False))
res.show()
res.explain(mode="formatted")

+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows
== Physical Plan ==
AdaptiveSparkPlan (31)
+- Sort (30)
   +- Exchange (29)
      +- HashAggregate (28)
         +- Exchange (27)
            +- HashAggregate (26)
               +- Project (25)
                  +- SortMergeJoin LeftOuter (24)
                     :- Sort (11)
                     :  +- Exchange (10)
                     :     +- Project (9)
                     :        +- SortMergeJoin LeftOuter (8)
                     :  

### Query 2.4

In [16]:
res = spark.sql(query_2_4(broadcast=False))
res.show()
res.explain(mode="formatted")

+---------+------------+-------------+--------------+
|   county|weather_cond|weather_hours|trips_per_hour|
+---------+------------+-------------+--------------+
|    Bronx|         dry|            2|           6.0|
|    Bronx|        rain|           10|           8.9|
|    Bronx|       humid|          210|          9.97|
|    Bronx|        cold|          173|         10.08|
|    Bronx|      stormy|          259|         10.45|
|    Bronx|    moderate|           24|         10.83|
| Brooklyn|        rain|           11|         23.45|
| Brooklyn|         dry|            2|          23.5|
| Brooklyn|    moderate|           26|         32.65|
| Brooklyn|       humid|          224|         33.55|
| Brooklyn|      stormy|          294|         34.36|
| Brooklyn|        cold|          186|         34.85|
|Manhattan|        rain|           11|        1719.0|
|Manhattan|    moderate|           26|       2757.77|
|Manhattan|         dry|            2|        3464.0|
|Manhattan|       humid|    

# Broadcasted queries
Make sure to manually check broadcast occurs in execution plan

### Query 2.1

In [17]:
result = spark.sql(query_2_1(broadcast=True))
result.show()
result.explain(mode="formatted")

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|  Van Cortlandt Park|    1|       18|
|                SoHo|    1|    20197|
|            Kips Bay|    1|    32984|
|Upper West Side S...|    1|    88474|
|Upper East Side S...|    1|   142708|
|       Rockaway Park|    1|       80|
|           Stapleton|    1|        4|
|East New York/Pen...|    1|      245|
|          Bath Beach|    1|       58|
|  Claremont/Bathgate|    1|      173|
|Bay Terrace/Fort ...|    1|       38|
|       Fordham South|    1|       91|
|             Bayside|    1|       81|
|    Garment District|    1|    48093|
|     Cambria Heights|    1|      143|
|    Bensonhurst East|    1|      145|
|         Great Kills|    1|        1|
|Upper West Side N...|    1|    64234|
|   Kew Gardens Hills|    1|      151|
|Springfield Garde...|    1|      446|
+--------------------+-----+---------+
only showing top 20 rows
== Physical Plan ==
AdaptiveSparkPlan (

### Query 2.2

In [18]:
result = spark.sql(query_2_2(broadcast=True))
result.show()
result.explain(mode="formatted")

+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 424773| 3.470743355446981|
|  zero_or_null|2539795|3.6824345463125323|
+--------------+-------+------------------+

== Physical Plan ==
AdaptiveSparkPlan (10)
+- HashAggregate (9)
   +- Exchange (8)
      +- HashAggregate (7)
         +- Project (6)
            +- BroadcastHashJoin LeftOuter BuildRight (5)
               :- Scan parquet spark_catalog.default.taxi_trips (1)
               +- BroadcastExchange (4)
                  +- Filter (3)
                     +- Scan parquet spark_catalog.default.weather (2)


(1) Scan parquet spark_catalog.default.taxi_trips
Output [2]: [pu_datetime#8312, trip_distance#8317]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/mac/Documents/dev/ID2221/dic/spark_project/spark-warehouse/taxi_trips]
ReadSchema: struct<pu_datetime:timestamp,trip_distance:float>

(2) Scan parquet spark_catalog.defa

### Query 2.3

In [19]:
res = spark.sql(query_2_3(broadcast=True))
res.show()
res.explain(mode="formatted")

+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows
== Physical Plan ==
AdaptiveSparkPlan (25)
+- Sort (24)
   +- Exchange (23)
      +- HashAggregate (22)
         +- Exchange (21)
            +- HashAggregate (20)
               +- Project (19)
                  +- BroadcastHashJoin LeftOuter BuildRight (18)
                     :- Project (6)
                     :  +- BroadcastHashJoin LeftOuter BuildRight (5)
                     :     :- Scan parquet spark_catalog.default.taxi_trips (1)
 

### Query 2.4

In [20]:
res = spark.sql(query_2_4(broadcast=True))
res.show()
res.explain(mode="formatted")

+---------+------------+-------------+--------------+
|   county|weather_cond|weather_hours|trips_per_hour|
+---------+------------+-------------+--------------+
|    Bronx|         dry|            2|           6.0|
|    Bronx|        rain|           10|           8.9|
|    Bronx|       humid|          210|          9.97|
|    Bronx|        cold|          173|         10.08|
|    Bronx|      stormy|          259|         10.45|
|    Bronx|    moderate|           24|         10.83|
| Brooklyn|        rain|           11|         23.45|
| Brooklyn|         dry|            2|          23.5|
| Brooklyn|    moderate|           26|         32.65|
| Brooklyn|       humid|          224|         33.55|
| Brooklyn|      stormy|          294|         34.36|
| Brooklyn|        cold|          186|         34.85|
|Manhattan|        rain|           11|        1719.0|
|Manhattan|    moderate|           26|       2757.77|
|Manhattan|         dry|            2|        3464.0|
|Manhattan|       humid|    